In [1]:
import numpy as np
import math
from numba import vectorize, cuda

# Set initial number of neurons for the first test
n = 1_000_000

# Generate initial data
greyscales = np.floor(np.random.uniform(0, 255, n).astype(np.float32))
weights = np.random.normal(.5, .1, n).astype(np.float32)

# Pack arguments for convenience
arguments = {
    "greyscales": greyscales,
    "weights": weights
}

In [2]:
# Pure NumPy implementation

def normalize_np(grayscales):
    return grayscales / 255.0

def weigh_np(values, weights):
    return values * weights

def activate_np(values):
    return (np.exp(values) - np.exp(-values)) / (np.exp(values) + np.exp(-values))

def create_hidden_layer_numpy(greyscales, weights):
    normalized = normalize_np(greyscales)
    weighted = weigh_np(normalized, weights)
    return activate_np(weighted)

In [3]:
# Numba CPU implementation

@vectorize(['float32(float32)'], target='cpu')
def normalize_cpu(grayscales):
    return grayscales / 255.0

@vectorize(['float32(float32, float32)'], target='cpu')
def weigh_cpu(values, weights):
    return values * weights

@vectorize(['float32(float32)'], target='cpu')
def activate_cpu(values):
    # Use math.exp instead of numpy.exp for Numba compatibility
    return (math.exp(values) - math.exp(-values)) / (math.exp(values) + math.exp(-values))

def create_hidden_layer_cpu(greyscales, weights):
    normalized = normalize_cpu(greyscales)
    weighted = weigh_cpu(normalized, weights)
    return activate_cpu(weighted)

In [4]:
# Numba GPU implementation with optimized data transfer

@vectorize(['float32(float32)'], target='cuda')
def normalize_gpu(grayscales):
    return grayscales / 255.0

@vectorize(['float32(float32, float32)'], target='cuda')
def weigh_gpu(values, weights):
    return values * weights

@vectorize(['float32(float32)'], target='cuda')
def activate_gpu(values):
    return (math.exp(values) - math.exp(-values)) / (math.exp(values) + math.exp(-values))

def create_hidden_layer_gpu(greyscales, weights):
    # Transfer arrays to device memory once to avoid PCIe bottleneck
    d_greyscales = cuda.to_device(greyscales)
    d_weights = cuda.to_device(weights)

    # Perform all calculations on the device
    d_normalized = normalize_gpu(d_greyscales)
    d_weighted = weigh_gpu(d_normalized, d_weights)
    d_activated = activate_gpu(d_weighted)

    # Copy the final result back to host memory
    return d_activated.copy_to_host()

In [5]:
# JIT Warm-up sequence
# Execute functions with a small subset to trigger compilation

warmup_g = greyscales[:10]
warmup_w = weights[:10]

_ = create_hidden_layer_numpy(warmup_g, warmup_w)
_ = create_hidden_layer_cpu(warmup_g, warmup_w)
_ = create_hidden_layer_gpu(warmup_g, warmup_w)

print("Warm-up complete. Functions are compiled.")

Warm-up complete. Functions are compiled.


/home/xairaven/Files/University/10/GPT/Lab4/project/.venv/lib64/python3.14/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/xairaven/Files/University/10/GPT/Lab4/project/.venv/lib64/python3.14/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/xairaven/Files/University/10/GPT/Lab4/project/.venv/lib64/python3.14/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [6]:
# Benchmark for n = 1M

print("NumPy version (1M):")
%timeit create_hidden_layer_numpy(**arguments)

print("\nNumba CPU version (1M):")
%timeit create_hidden_layer_cpu(**arguments)

print("\nNumba GPU version (1M):")
%timeit create_hidden_layer_gpu(**arguments)

NumPy version (1M):
9.74 ms ± 257 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)

Numba CPU version (1M):
8.89 ms ± 246 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)

Numba GPU version (1M):
3.88 ms ± 120 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
# Generate data and benchmark for n = 100M

n_large = 100_000_000

greyscales_large = np.floor(np.random.uniform(0, 255, n_large).astype(np.float32))
weights_large = np.random.normal(.5, .1, n_large).astype(np.float32)

arguments_large = {
    "greyscales": greyscales_large,
    "weights": weights_large
}

print("NumPy version (100M):")
%timeit create_hidden_layer_numpy(**arguments_large)

print("\nNumba CPU version (100M):")
%timeit create_hidden_layer_cpu(**arguments_large)

print("\nNumba GPU version (100M):")
%timeit create_hidden_layer_gpu(**arguments_large)

NumPy version (100M):
1.87 s ± 178 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Numba CPU version (100M):
970 ms ± 46.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Numba GPU version (100M):
295 ms ± 8.88 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
